## 准备数据

In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [3]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [10]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        self.W1=tf.Variable(tf.random.normal([784, 128], stddev=0.1))
        self.b1=tf.Variable(tf.zeros([128]))
        self.W2=tf.Variable(tf.random.normal([128, 10], stddev=0.1))
        self.b2=tf.Variable(tf.zeros([10]))
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        x_flat=tf.reshape(x, [-1, 784])
        h=tf.matmul(x_flat, self.W1)+self.b1
        h=tf.nn.tanh(h)
        logits=tf.matmul(h, self.W2)+self.b2
        ####################
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [6]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [14]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 1.341443 ; accuracy 0.6616
epoch 1 : loss 1.337357 ; accuracy 0.66298336
epoch 2 : loss 1.3333 ; accuracy 0.66445
epoch 3 : loss 1.3292714 ; accuracy 0.6656833
epoch 4 : loss 1.3252711 ; accuracy 0.66695
epoch 5 : loss 1.3212987 ; accuracy 0.6683667
epoch 6 : loss 1.3173542 ; accuracy 0.6697
epoch 7 : loss 1.3134369 ; accuracy 0.67125
epoch 8 : loss 1.309547 ; accuracy 0.6724833
epoch 9 : loss 1.305684 ; accuracy 0.6738667
epoch 10 : loss 1.3018477 ; accuracy 0.67516667
epoch 11 : loss 1.2980379 ; accuracy 0.6764
epoch 12 : loss 1.2942543 ; accuracy 0.6777
epoch 13 : loss 1.2904967 ; accuracy 0.6788333
epoch 14 : loss 1.286765 ; accuracy 0.6798667
epoch 15 : loss 1.2830588 ; accuracy 0.6813167
epoch 16 : loss 1.2793778 ; accuracy 0.68233335
epoch 17 : loss 1.2757219 ; accuracy 0.6835
epoch 18 : loss 1.2720908 ; accuracy 0.68476665
epoch 19 : loss 1.2684844 ; accuracy 0.68626666
epoch 20 : loss 1.2649024 ; accuracy 0.68765
epoch 21 : loss 1.2613446 ; accuracy 0.68873334
e